<a href="https://colab.research.google.com/github/juanmg1984/infovis/blob/main/data360_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración Profunda: API Data360 del Banco Mundial

Este cuaderno profundiza en la estructura de la API **Data360**, analizando los metadatos disponibles: cobertura temporal, áreas geográficas y categorías de datos (temas).

**API Endpoint**: `https://data360api.worldbank.org`  
**Documentación**: Data360 Open API Spec

In [1]:
import requests
import pandas as pd
import duckdb
import altair as alt
import json

# Configuración base
BASE_URL = "https://data360api.worldbank.org/data360"
# alt.renderers.enable('colab') # Descomentar si se usa en Google Colab

## 1. Cobertura Temporal

¿Qué años de información podemos extraer? Analizaremos el rango temporal disponible para un indicador global representativo.

In [2]:
def get_time_range(indicator_id="WB_WDI_SP_POP_TOTL", area="WLD"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": area}
    resp = requests.get(url, params=params).json()
    years = [int(v['TIME_PERIOD']) for v in resp['value'] if v['TIME_PERIOD'].isdigit()]
    return min(years), max(years), len(years)

min_y, max_y, total_y = get_time_range()
print(f"Rango temporal disponible: {min_y} - {max_y} ({total_y} años)")

Rango temporal disponible: 1960 - 2024 (65 años)


## 2. Cobertura Geográfica (Países y Áreas)

La API no solo incluye países, sino también agregados regionales (ej. América Latina, Europa) y niveles de ingreso (ej. Países de ingreso bajo).

In [3]:
def get_available_areas(indicator_id="WB_WDI_SP_POP_TOTL"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "isLatestData": True}
    resp = requests.get(url, params=params).json()
    areas = [v['REF_AREA'] for v in resp['value']]
    return sorted(list(set(areas)))

areas = get_available_areas()
print(f"Total de áreas/países disponibles: {len(areas)}")
print("Muestra de códigos de área (ISO3):", areas[:25])

Total de áreas/países disponibles: 265
Muestra de códigos de área (ISO3): ['ABW', 'AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH']


## 3. Categorías de Datos (Temas)

Los indicadores están agrupados por temas. Utilizaremos el endpoint de metadatos para descubrir las categorías principales.

In [4]:
def get_topics(top=200):
    url = f"{BASE_URL}/metadata"
    query = {"query": f"&$filter=series_description/database_id eq 'WB_WDI'&$select=series_description/topics/name&$top={top}"}
    resp = requests.post(url, json=query).json()

    unique_topics = set()
    if 'value' in resp:
        for item in resp['value']:
            sd = item.get('series_description')
            if sd is None:
                continue # Skip items where series_description is None

            # La API puede devolver series_description como dict o list segun el dataset
            # Ensure entries contains only dictionaries, filtering out any None or other types
            if isinstance(sd, list):
                entries = [e for e in sd if isinstance(e, dict)]
            elif isinstance(sd, dict):
                entries = [sd]
            else:
                entries = [] # Handle unexpected type for sd, skip if not dict or list

            for entry in entries:
                # entry is guaranteed to be a dict here
                for t in entry.get('topics', []):
                    # Ensure t is a dictionary before calling .get()
                    if isinstance(t, dict) and t.get('name'):
                        unique_topics.add(t['name'])
    return sorted(list(unique_topics))

topics = get_topics()
print(f"Temas principales detectados ({len(topics)}):")
for t in topics: print(f"- {t}")

Temas principales detectados (42):
- Agriculture and Food
- Carbon Pricing, Climate Finance and GHG accounting
- Climate Change
- Connectivity
- Crisis and Disaster Risk Finance
- Digital
- Digital Services
- Economic Policy
- Education
- Energy & Extractives
- Environment
- Finance
- Food and Nutrition Security
- Gender
- Global Infrastructure Finance
- Growth
- Growth and Jobs
- Health, Nutrition & Population
- Housing
- Inequality and Shared Prosperity
- Infrastructure
- Institutions
- Investment and Business Climate
- Jobs
- Learning
- Mortality
- People
- Planet
- Poverty
- Prosperity
- Schooling
- Social Protection
- Strengthening Agi Food Systems
- Teaching
- Trade Outcomes
- Trade, Investment and Competitiveness
- Transmission and Distribution
- Transport
- Trends in the Determinants of Food Security Outcomes
- Urban, Resilience and Land
- Water
- Water and the Economy


## 4. Visualización Exploratoria

### Crecimiento Comparativo en el Cono Sur
Utilizando los datos descubiertos, comparamos el crecimiento poblacional relativo entre países vecinos.

In [5]:
def get_indicator_data(indicator, countries):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator, "REF_AREA": ",".join(countries)}
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp['value'])
    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'])
    df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'])
    return df

df_cono_sur = get_indicator_data("WB_WDI_SP_POP_TOTL", ["ARG", "CHL", "URY", "BRA"])

# Uso de DuckDB para filtrar o procesar si fuera necesario
df_filtered = duckdb.query("SELECT * FROM df_cono_sur WHERE TIME_PERIOD >= 1960").df()

# Visualización interactiva con Altair
line_chart = alt.Chart(df_filtered).mark_line(point=True).encode(
    x=alt.X('TIME_PERIOD:O', title='Año'),
    y=alt.Y('OBS_VALUE:Q', title='Población', scale=alt.Scale(type='log')),
    color=alt.Color('REF_AREA:N', title='País'),
    tooltip=['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
).properties(
    title='Población Total en el Cono Sur (1960-2023)',
    width=600,
    height=300
).interactive()

line_chart

alt.Chart(...)

### 5. Preparación de Datos para PCA: Agrupando Países por Indicadores

Para realizar un PCA y visualizar países en un plano de dos dimensiones, necesitamos un conjunto de indicadores diversos para varios países. Seleccionaremos algunos indicadores representativos y un grupo más amplio de países para este análisis.

In [6]:
# Definir una lista de indicadores relevantes para PCA
# Usamos algunos de los que ya hemos visto y añadimos otros para mayor diversidad
indicators_for_pca = {
    "WB_WDI_SP_POP_TOTL": "Poblacion Total", # Population, total
    "WB_WDI_NY_GDP_PCAP_CD": "PIB per Capita (USD)", # GDP per capita (current US$)
    "WB_WDI_SP_DYN_LE00_IN": "Esperanza de Vida al Nacer", # Life expectancy at birth, total (years)
    "WB_WDI_SE_ADT_LITR_ZS": "Tasa de Alfabetizacion", # Literacy rate, adult total (% of people ages 15 and above)
    "WB_WDI_EN_ATM_CO2E_PC": "Emisiones CO2 per Capita", # CO2 emissions (metric tons per capita)
    "WB_WDI_SH_MED_BEDS_ZS": "Camas Hospitalarias (per 1k personas)", # Hospital beds (per 1,000 people)
    "WB_WDI_IT_NET_USER_ZS": "Usuarios de Internet (% poblacion)" # Internet users (% of population)
}

# Seleccionar un conjunto de países para el análisis (ej. todos los de América Latina y el Caribe, y algunos de otras regiones)
# Usaremos los códigos ISO3 disponibles en la API
countries_for_pca = [
    'ARG', 'BRA', 'CHL', 'COL', 'MEX', 'PER', 'URY', 'VEN', # LatAm
    'CAN', 'USA', # North America
    'DEU', 'FRA', 'GBR', 'ITA', 'ESP', # Europe
    'CHN', 'IND', 'JPN', 'AUS', # Asia/Oceania
    'ZAF', 'NGA' # Africa
]

# Funcion para obtener datos de un solo indicador para multiples areas y un año específico
def get_indicator_data_for_year(indicator_id, areas, year=None):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": ",".join(areas)}
    if year: # Intentar obtener el dato del año especificado, o el más cercano si no hay
        params['TIME_PERIOD'] = str(year)
        # params['isLatestData'] = True # Esto tomaría el último disponible si no hay para el año exacto
    else:
        params['isLatestData'] = True # Si no se especifica año, tomar el más reciente disponible

    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp.get('value', []))

    # Asegurarse de que REF_AREA y OBS_VALUE estén presentes y limpios
    if df.empty:
        return pd.DataFrame()

    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    df = df.dropna(subset=['OBS_VALUE'])

    # Si hay múltiples entradas por país (ej. diferentes años si no se usó isLatestData o un año específico)
    # nos quedamos con el valor más reciente o el que más se ajuste al año solicitado
    if 'TIME_PERIOD' in df.columns and not year: # Si no se pidió un año específico y se usó isLatestData
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        df = df.sort_values(by='TIME_PERIOD', ascending=False).drop_duplicates(subset=['REF_AREA'])
    elif 'TIME_PERIOD' in df.columns and year: # Si se pidió un año específico
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        # Intentar obtener el dato del año exacto, si no, el más cercano posterior, luego el más cercano anterior
        df['time_diff'] = abs(df['TIME_PERIOD'] - year)
        df = df.sort_values(by='time_diff').drop_duplicates(subset=['REF_AREA'])

    return df[['REF_AREA', 'OBS_VALUE']].rename(columns={'OBS_VALUE': indicator_id})

# Recopilar los datos para PCA
df_pca_input = pd.DataFrame({'REF_AREA': countries_for_pca})

for indicator_code, indicator_name in indicators_for_pca.items():
    print(f"Obteniendo datos para: {indicator_name} ({indicator_code})...")
    temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca, year=2022) # Intentar obtener datos de 2022
    if not temp_df.empty:
        df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
    else:
        print(f"  No se encontraron datos para {indicator_name} en 2022. Intentando con datos más recientes...")
        temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca) # Obtener el más reciente si 2022 no funciona
        if not temp_df.empty:
             df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
        else:
            print(f"  No se encontraron datos recientes para {indicator_name}.")

# Establecer REF_AREA como índice para facilitar el PCA
df_pca_input = df_pca_input.set_index('REF_AREA')

# Mostrar las primeras filas del DataFrame listo para PCA
print("\nDataFrame preparado para PCA:")
display(df_pca_input.head())

Obteniendo datos para: Poblacion Total (WB_WDI_SP_POP_TOTL)...
Obteniendo datos para: PIB per Capita (USD) (WB_WDI_NY_GDP_PCAP_CD)...
Obteniendo datos para: Esperanza de Vida al Nacer (WB_WDI_SP_DYN_LE00_IN)...
Obteniendo datos para: Tasa de Alfabetizacion (WB_WDI_SE_ADT_LITR_ZS)...
Obteniendo datos para: Emisiones CO2 per Capita (WB_WDI_EN_ATM_CO2E_PC)...
  No se encontraron datos para Emisiones CO2 per Capita en 2022. Intentando con datos más recientes...
  No se encontraron datos recientes para Emisiones CO2 per Capita.
Obteniendo datos para: Camas Hospitalarias (per 1k personas) (WB_WDI_SH_MED_BEDS_ZS)...
  No se encontraron datos para Camas Hospitalarias (per 1k personas) en 2022. Intentando con datos más recientes...
Obteniendo datos para: Usuarios de Internet (% poblacion) (WB_WDI_IT_NET_USER_ZS)...

DataFrame preparado para PCA:


,WB_WDI_SP_POP_TOTL,WB_WDI_NY_GDP_PCAP_CD,WB_WDI_SP_DYN_LE00_IN,WB_WDI_SE_ADT_LITR_ZS,WB_WDI_SH_MED_BEDS_ZS,WB_WDI_IT_NET_USER_ZS
REF_AREA,,,,,,
ARG,45407904,13962.189409,75.806,NaN,3.34,88.3754
BRA,210306415,9281.332821,74.872,94.385387,2.46,80.5278
CHL,19553036,15405.616577,79.176,NaN,1.97,93.5377
COL,51737944,6680.445069,76.508,95.313883,1.69,72.7964
MEX,128613117,11405.794047,73.973,95.848660,1.02,78.6322


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Manejo de Valores Faltantes
df_pca_cleaned = df_pca_input.dropna(axis=1, how='all')
df_pca_cleaned = df_pca_cleaned.dropna(axis=0)

print(f"DataFrame después de manejar valores faltantes: {df_pca_cleaned.shape[0]} países, {df_pca_cleaned.shape[1]} indicadores.")

if df_pca_cleaned.empty or df_pca_cleaned.shape[1] < 2:
    print("No hay suficientes datos o indicadores después de la limpieza para realizar PCA.")
else:
    # 2. Escalado y PCA
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_pca_cleaned)

    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(scaled_data)

    pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'], index=df_pca_cleaned.index).reset_index()

    # Recuperar algunos indicadores para los tooltips usando DuckDB
    original_data = df_pca_cleaned.reset_index()
    # Unimos con DuckDB para demostrar su uso
    pca_final = duckdb.query("SELECT p.*, o.* EXCLUDE (REF_AREA) FROM pca_df p JOIN original_data o ON p.REF_AREA = o.REF_AREA").df()

    # 3. Visualización Interactiva con Altair
    selection = alt.selection_point(fields=['REF_AREA'], on='click')

    chart = alt.Chart(pca_final).mark_circle(size=100).encode(
        x=alt.X('PC1:Q', title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)'),
        y=alt.Y('PC2:Q', title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)'),
        color=alt.condition(selection, alt.value('steelblue'), alt.value('lightgray')),
        tooltip=['REF_AREA', 'PC1', 'PC2'] + [col for col in df_pca_cleaned.columns]
    ).add_params(
        selection
    ).properties(
        title='PCA Interactivo de Países (Indicadores Banco Mundial)',
        width=600,
        height=400
    ).interactive()

    # Añadir etiquetas de texto
    text = chart.mark_text(
        align='left',
        baseline='middle',
        dx=7
    ).encode(
        text='REF_AREA:N'
    )

    display(chart + text)

DataFrame después de manejar valores faltantes: 5 países, 6 indicadores.


alt.LayerChart(...)

### Siguientes Pasos: Escalado, PCA y Visualización

Ahora que tenemos el DataFrame `df_pca_input`, los siguientes pasos serían:

1.  **Manejo de Valores Faltantes**: Imputar o eliminar filas/columnas con datos faltantes. (Aunque la función `get_indicator_data_for_year` ya los maneja un poco).
2.  **Escalado de Datos**: Los indicadores tienen diferentes unidades y escalas, por lo que es crucial escalarlos (ej. con `StandardScaler` de `sklearn.preprocessing`) antes de aplicar PCA.
3.  **Aplicar PCA**: Utilizar `PCA` de `sklearn.decomposition` para reducir la dimensionalidad a 2 componentes principales.
4.  **Visualización**: Graficar los países en un diagrama de dispersión usando las dos primeras componentes principales, lo que permitirá identificar grupos de países con características similares.

## Resumen Técnico de la Exploración

- **Rango Temporal**: 1960 a 2024. Ideal para series de largo plazo.
- **Cobertura Espacial**: 265 códigos de área, permitiendo análisis país por país o por bloques económicos (G20, OCDE, etc.).
- **Temáticas**: Diversidad desde agricultura y medio ambiente hasta políticas económicas y agua.
- **Facilidad de Uso**: La API permite filtrado complejo mediante OData (usado en el endpoint de metadatos).